In [1]:
import pandas as pd
import numpy as np
import pathlib as pathlib

In [2]:
def find_project_root(marker: str = ".git") -> pathlib.Path:
    """Walk upward from the current working directory until a folder containing `marker` is found."""
    current = pathlib.Path.cwd()
    for candidate in [current, *current.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find project root (no '{marker}' found in {current} or any parent directory)"
    )


PROJECT_ROOT = find_project_root()
print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\benwa\OneDrive\Desktop\EnergyIQ


In [3]:
data_path = PROJECT_ROOT / "data"
time_segments = ["sept2024-feb2025", "march2025-aug2025", "sept2025-feb2026", "march2026-sept2026"]

all_dfs = []
for subfolder_name in time_segments:
    subfolder_path = data_path / subfolder_name
    csv_files = list(subfolder_path.glob("*.csv"))
    
    for csv_file in csv_files:
        file_path = subfolder_path / csv_file
        df = pd.read_csv(file_path)

        print(f"Data from {file_path}:")
        print(df.head())

        all_dfs.append(df)

energy_df = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()
energy_df = energy_df.sort_values('readingtime')
energy_df = energy_df.dropna(subset=['readingvalue'])

Data from c:\Users\benwa\OneDrive\Desktop\EnergyIQ\data\sept2024-feb2025\ohio_stadium_sept2024-feb2025.csv:
   meterid      sitename  simscode      utility          readingtime  \
0   246215  Ohio Stadium        82  ELECTRICITY  2024-09-01T05:00:00   
1   246214  Ohio Stadium        82  ELECTRICITY  2024-09-01T05:00:00   
2   246216  Ohio Stadium        82  ELECTRICITY  2024-09-01T05:00:00   
3   246218  Ohio Stadium        82  ELECTRICITY  2024-09-01T05:00:00   
4   246220  Ohio Stadium        82  ELECTRICITY  2024-09-01T05:00:00   

   readingvalue readingunits  year  month  day  
0     15.160404          kWh  2024      9    1  
1     23.561469          kWh  2024      9    1  
2     33.143207          kWh  2024      9    1  
3     43.121924          kWh  2024      9    1  
4     60.500417          kWh  2024      9    1  
Data from c:\Users\benwa\OneDrive\Desktop\EnergyIQ\data\sept2024-feb2025\scott_house_sept2024-feb2025.csv:
   meterid     sitename  simscode      utility          re

In [4]:
# number of readings and number of meters per sitename + utility group
energy_df.groupby(['sitename', 'utility'])['meterid'].agg(['count', 'nunique']).rename(
    columns={'count': 'n_readings', 'nunique': 'n_meters'}
)


n_readings  n_meters
sitename                  utility                          
Ohio Stadium              ELECTRICITY      501120         9
                          HEAT              55676         1
Scott House               COOLING           68693         1
                          ELECTRICITY      137394         2
                          GAS               68697         1
                          HEAT              68693         1
Thompson Memorial Library COOLING           55676         1
                          ELECTRICITY      111360         2
                          GAS               55680         1
                          HEAT              55676         1

In [5]:
# number of readings per meter
energy_df.groupby(['sitename', 'utility', 'meterid'])['readingvalue'].agg(['count']).rename(
    columns={'count': 'n_readings'}
).reset_index()

,sitename,utility,meterid,n_readings
0,Ohio Stadium,ELECTRICITY,246213,55680
1,Ohio Stadium,ELECTRICITY,246214,55680
2,Ohio Stadium,ELECTRICITY,246215,55680
3,Ohio Stadium,ELECTRICITY,246216,55680
4,Ohio Stadium,ELECTRICITY,246217,55680
5,Ohio Stadium,ELECTRICITY,246218,55680
6,Ohio Stadium,ELECTRICITY,246219,55680
7,Ohio Stadium,ELECTRICITY,246220,55680
8,Ohio Stadium,ELECTRICITY,246221,55680
9,Ohio Stadium,HEAT,247516,55676


In [6]:
data_path = PROJECT_ROOT / "data"
folder_name = "weather_data"

all_dfs = []
subfolder_path = data_path / folder_name
csv_files = list(subfolder_path.glob("*.csv"))

for csv_file in csv_files:
    file_path = subfolder_path / csv_file
    df = pd.read_csv(file_path)

    print(f"Data from {file_path}:")
    print(df.head())

    all_dfs.append(df)

weather_df = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()

Data from c:\Users\benwa\OneDrive\Desktop\EnergyIQ\data\weather_data\weather_sept2024-aug2025.csv:
                  date   latitude  longitude  temperature_2m  \
0  2024-09-01T00:00:00  40.079516 -83.073213       77.559799   
1  2024-09-01T01:00:00  40.079516 -83.073213       73.869797   
2  2024-09-01T02:00:00  40.079516 -83.073213       71.799797   
3  2024-09-01T03:00:00  40.079516 -83.073213       71.799797   
4  2024-09-01T04:00:00  40.079516 -83.073213       69.909805   

   shortwave_radiation  direct_radiation  diffuse_radiation  \
0                   31               0.0                 31   
1                    0               0.0                  0   
2                    0               0.0                  0   
3                    0               0.0                  0   
4                    0               0.0                  0   

   direct_normal_irradiance  relative_humidity_2m  dew_point_2m  \
0                       0.0             73.400024     68.379799   
1  

In [7]:
# we only have 1 location that we read from
weather_df.groupby(['latitude', 'longitude']).size().reset_index(name='n_readings')

,latitude,longitude,n_readings
0,40.079516,-83.073213,17424


In [14]:
energy_df.head(20)
weather_df.count()

date                        17424
latitude                    17424
longitude                   17424
temperature_2m              17424
shortwave_radiation         17424
direct_radiation            17424
diffuse_radiation           17424
direct_normal_irradiance    17424
relative_humidity_2m        17424
dew_point_2m                17424
precipitation               17424
wind_speed_10m              17424
wind_speed_100m             17424
wind_direction_100m         17424
wind_direction_10m          17424
cloud_cover                 17424
apparent_temperature        17424
partition_0                 17424
time                        17424
dtype: int64

In [15]:
# join 
energy_df['readingtime'] = pd.to_datetime(energy_df['readingtime'])
weather_df['time'] = pd.to_datetime(weather_df['date']) 

# Floor energy timestamps down to the hour
energy_df['hour'] = energy_df['readingtime'].dt.floor('h')

# Merge — each energy row picks up the weather row for its containing hour
merged_df = energy_df.merge(
    weather_df,
    left_on='hour',
    right_on='date',
    how='inner'
)
merged_df

ValueError: You are trying to merge on datetime64[ns] and object columns for key 'hour'. If you wish to proceed you should use pd.concat